# Chapter 6 — Policy Gradients, PPO & DPO## Concept Notebook[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)**GPU optional. Runtime: ~30 minutes.**Implements REINFORCE, GAE, PPO-Clip, and a minimal DPO loss — the algorithmic core of modern LLM training.

In [ ]:
import torchimport torch.nn as nnimport numpy as npimport matplotlib.pyplot as pltprint('PyTorch:', torch.__version__)

## 6.1 REINFORCE: The Policy Gradient TheoremThe policy gradient: ∇J(θ) = E[∇ log π_θ(a|s) · G_t]Where G_t is the discounted return. We maximise expected return by gradient ascent.

In [ ]:
class PolicyNetwork(nn.Module):    def __init__(self, obs_dim, n_actions):        super().__init__()        self.net = nn.Sequential(            nn.Linear(obs_dim, 64), nn.Tanh(),            nn.Linear(64, 64), nn.Tanh(),            nn.Linear(64, n_actions)        )    def forward(self, x):        return torch.distributions.Categorical(logits=self.net(x))def compute_returns(rewards, gamma=0.99):    """Compute discounted returns G_t = sum_{k=0}^{T-t} γ^k r_{t+k}"""    G, returns = 0, []    for r in reversed(rewards):        G = r + gamma * G        returns.insert(0, G)    returns = torch.tensor(returns, dtype=torch.float32)    # Normalise for stable training    returns = (returns - returns.mean()) / (returns.std() + 1e-8)    return returnsprint('REINFORCE components ready')

## 6.2 Generalised Advantage Estimation (GAE)GAE interpolates between high-variance Monte Carlo (λ=1) and high-bias TD (λ=0):A_t^GAE = Σ_{l≥0} (γλ)^l δ_{t+l} where δ_t = r_t + γV(s_{t+1}) - V(s_t)

In [ ]:
def compute_gae(rewards, values, next_values, dones, gamma=0.99, lam=0.95):    """    GAE advantage estimation.    rewards, values, next_values, dones: all tensors of length T    """    T = len(rewards)    advantages = torch.zeros(T)    gae = 0.0    for t in reversed(range(T)):        next_val = 0.0 if dones[t] else next_values[t]        delta = rewards[t] + gamma * next_val - values[t]        gae = delta + gamma * lam * (0.0 if dones[t] else gae)        advantages[t] = gae    return (advantages - advantages.mean()) / (advantages.std() + 1e-8)print('GAE implementation ready')

## 6.3 PPO-Clip: The Most Important RL UpdatePPO prevents large policy updates by clipping the probability ratio:L^CLIP = E[min(r(θ)A, clip(r(θ), 1-ε, 1+ε)A)]where r(θ) = π_θ(a|s) / π_θ_old(a|s)

In [ ]:
def ppo_loss(log_probs_new, log_probs_old, advantages, epsilon=0.2):    """PPO-Clip objective (to be maximised)."""    ratio = torch.exp(log_probs_new - log_probs_old)    clipped = torch.clamp(ratio, 1 - epsilon, 1 + epsilon)    loss = -torch.min(ratio * advantages, clipped * advantages).mean()    return loss# Demonstrate clipping effectratios = torch.linspace(0.5, 2.0, 100)advantage_pos = torch.ones(100) * 1.0  # positive advantageadvantage_neg = torch.ones(100) * -1.0  # negative advantageL_pos = torch.min(ratios * advantage_pos, torch.clamp(ratios, 0.8, 1.2) * advantage_pos)L_neg = torch.min(ratios * advantage_neg, torch.clamp(ratios, 0.8, 1.2) * advantage_neg)fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))ax1.plot(ratios.numpy(), L_pos.numpy(), 'b-', linewidth=2, label='PPO-Clip')ax1.plot(ratios.numpy(), (ratios * advantage_pos).numpy(), 'r--', alpha=0.5, label='Unclipped')ax1.axvline(0.8, color='gray', linestyle=':')ax1.axvline(1.2, color='gray', linestyle=':')ax1.set_title('Positive Advantage: Clips Upper Ratio')ax1.legend()ax2.plot(ratios.numpy(), L_neg.numpy(), 'b-', linewidth=2, label='PPO-Clip')ax2.plot(ratios.numpy(), (ratios * advantage_neg).numpy(), 'r--', alpha=0.5, label='Unclipped')ax2.set_title('Negative Advantage: Clips Lower Ratio')ax2.legend()plt.suptitle('PPO Clipping Effect (ε=0.2)', fontsize=13)plt.tight_layout()plt.show()

## 6.4 DPO Loss: Preference Optimisation Without a Reward ModelDPO directly optimises preferences:L_DPO = -E[log σ(β·(log(π(y_w)/π_ref(y_w)) - log(π(y_l)/π_ref(y_l))))]This is Chapter 16's algorithm derived here for the first time.

In [ ]:
def dpo_loss(policy_chosen_logps, policy_rejected_logps,             ref_chosen_logps, ref_rejected_logps, beta=0.1):    chosen_logratios   = policy_chosen_logps   - ref_chosen_logps    rejected_logratios = policy_rejected_logps - ref_rejected_logps    logits = beta * (chosen_logratios - rejected_logratios)    loss = -torch.nn.functional.logsigmoid(logits).mean()    accuracy = (chosen_logratios > rejected_logratios).float().mean()    return loss, accuracy# Demonstrate DPO with synthetic log-probabilitiesbatch_size = 32torch.manual_seed(42)policy_chosen   = torch.randn(batch_size) * 0.5 - 1.0policy_rejected = torch.randn(batch_size) * 0.5 - 2.0ref_chosen      = torch.randn(batch_size) * 0.3 - 1.5ref_rejected    = torch.randn(batch_size) * 0.3 - 1.5loss, acc = dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected)print(f'DPO Loss: {loss.item():.4f}')print(f'Reward accuracy (chosen > rejected): {acc.item()*100:.1f}%')

## ✅ Key Takeaways1. REINFORCE: gradient ascent on log-prob × return — correct but high variance2. GAE (λ=0.95): balances variance and bias, feeds into PPO3. PPO-Clip: prevents destructive updates by constraining the policy change per step4. DPO: eliminates the reward model by directly optimising the preference distribution